In [33]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [34]:
df = pd.read_csv("ecommerce_customer_behavior_dataset_v2.csv")

In [35]:
print(df.shape)
print(df.columns)
df.head()

(17049, 18)
Index(['Order_ID', 'Customer_ID', 'Date', 'Age', 'Gender', 'City',
       'Product_Category', 'Unit_Price', 'Quantity', 'Discount_Amount',
       'Total_Amount', 'Payment_Method', 'Device_Type',
       'Session_Duration_Minutes', 'Pages_Viewed', 'Is_Returning_Customer',
       'Delivery_Time_Days', 'Customer_Rating'],
      dtype='object')


,Order_ID,Customer_ID,Date,Age,Gender,City,Product_Category,Unit_Price,Quantity,Discount_Amount,Total_Amount,Payment_Method,Device_Type,Session_Duration_Minutes,Pages_Viewed,Is_Returning_Customer,Delivery_Time_Days,Customer_Rating
0,ORD_000001-1,CUST_00001,2023-05-29,40,Male,Ankara,Books,29.18,1,0.00,29.18,Digital Wallet,Mobile,14,9,True,13,4
1,ORD_000001-2,CUST_00001,2023-10-12,40,Male,Ankara,Home & Garden,644.40,1,138.05,506.35,Credit Card,Desktop,14,8,True,6,2
2,ORD_000001-3,CUST_00001,2023-12-05,40,Male,Ankara,Sports,332.82,5,0.00,1664.10,Credit Card,Mobile,15,10,True,9,4
3,ORD_000002-1,CUST_00002,2023-05-11,33,Male,Istanbul,Food,69.30,5,71.05,275.45,Digital Wallet,Desktop,16,13,True,4,4
4,ORD_000002-2,CUST_00002,2023-06-16,33,Male,Istanbul,Beauty,178.15,3,0.00,534.45,Credit Card,Mobile,14,7,True,6,4


In [36]:
df.columns = (df.columns.str.strip().str.lower())

In [37]:
#Check and remove duplicate rows
df = df.drop_duplicates()

In [38]:
#Convert date column 
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])

In [39]:
#Validate and fix data types
df["age"] = df["age"].astype(int)
df["quantity"] = df["quantity"].astype(int)
df["unit_price"] = df["unit_price"].astype(float)
df["discount_amount"] = df["discount_amount"].astype(float)
df["total_amount"] = df["total_amount"].astype(float)
df["session_duration_minutes"] = df["session_duration_minutes"].astype(int)
df["pages_viewed"] = df["pages_viewed"].astype(int)
df["delivery_time_days"] = df["delivery_time_days"].astype(int)
df["customer_rating"] = df["customer_rating"].astype(int)

In [40]:
#Handle missing values
df.isnull().sum().sort_values(ascending=False)
df = df.dropna(subset=["order_id", "customer_id"])

In [41]:
#Handling missing numerical values
num_cols = [
    "unit_price", "quantity", "discount_amount",
    "total_amount", "session_duration_minutes",
    "pages_viewed", "delivery_time_days", "customer_rating"
]

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

In [42]:
#Handling missing categorical values
cat_cols = [
    "gender", "city", "product_category",
    "payment_method", "device_type"
]

for col in cat_cols:
    df[col] = df[col].fillna("Unknown")

In [43]:
#Filter transactions with zero or negative prices
df = df[
    (df["unit_price"] > 0) &
    (df["total_amount"] > 0)
]

In [44]:
#Basket size per order
basket_size = (
    df.groupby("order_id")["quantity"]
    .sum()
    .reset_index(name="basket_size")
)

df = df.merge(basket_size, on="order_id", how="left")

In [45]:
#Order value (transaction value)
order_value = (
    df.groupby("order_id")["total_amount"]
    .sum()
    .reset_index(name="order_value")
)

df = df.merge(order_value, on="order_id", how="left")

In [46]:
#Customer-level aggregation
customer_summary = (
    df.groupby("customer_id")
    .agg(
        total_orders=("order_id", "nunique"),
        total_spend=("total_amount", "sum"),
        avg_basket_size=("basket_size", "mean"),
        avg_order_value=("order_value", "mean"),
        last_purchase_date=("date", "max"),
        returning_customer=("is_returning_customer", "max")
    )
    .reset_index()
)

In [47]:
df.describe()
customer_summary.head()

,customer_id,total_orders,total_spend,avg_basket_size,avg_order_value,last_purchase_date,returning_customer
0,CUST_00001,3,2199.63,2.333333,733.210000,2023-12-05,True
1,CUST_00002,2,809.90,4.000000,404.950000,2023-06-16,True
2,CUST_00003,2,3030.81,3.500000,1515.405000,2024-01-03,True
3,CUST_00004,1,383.22,5.000000,383.220000,2024-02-13,False
4,CUST_00005,3,2422.73,2.666667,807.576667,2023-06-21,True


In [48]:
print(customer_summary)

     customer_id  total_orders  total_spend  avg_basket_size  avg_order_value  \
0     CUST_00001             3      2199.63         2.333333       733.210000   
1     CUST_00002             2       809.90         4.000000       404.950000   
2     CUST_00003             2      3030.81         3.500000      1515.405000   
3     CUST_00004             1       383.22         5.000000       383.220000   
4     CUST_00005             3      2422.73         2.666667       807.576667   
...          ...           ...          ...              ...              ...   
4995  CUST_04996             4      3001.96         3.250000       750.490000   
4996  CUST_04997             4     15440.42         3.250000      3860.105000   
4997  CUST_04998             1       482.90         5.000000       482.900000   
4998  CUST_04999             1       137.30         2.000000       137.300000   
4999  CUST_05000             4      7282.41         2.500000      1820.602500   

     last_purchase_date  re

In [49]:
df.to_csv("cleaned_transaction_data.csv", index=False)

In [50]:
customer_summary.to_csv("customer_summary_data.csv", index=False)